# Реранк кандидатов: признаки и линейные бусты

## Цель

Добавим сигналы, которых BM25 не видит, чтобы поднять score (все подтверждены в 01_eda):

- **локация** — 83% релевантных пар совпадают по `location_id`;
- **фильтры поиска** — токены `search_infm_params_text` почти полностью входят в параметры выбранного объявления (overlap 0.97);
- **покрытие заголовка** — запросы короткие и лексически близки к заголовкам;
- **рейтинг / отзывы / популярность** — пользователи чаще выбирают проверенных исполнителей.

Схема: `BM25 top-200 -> матрица признаков на пару (запрос, кандидат) -> линейный скор -> top-50`

## Подготовка

### Импорты и константы

In [14]:
import gc

import numpy as np
import pandas as pd

from avito_candidate_gen.data import (
    train_val_split,
    DATA_PROCESSED_DIR,
    EXPERIMENTS_DIR,
)
from avito_candidate_gen.preprocessing import (
    add_title_lemma_column,
    build_query_text,
)
from avito_candidate_gen.retrieval.bm25 import BM25Retriever
from avito_candidate_gen.features import (
    FEATURE_COLUMNS,
    FeatureBuilder,
    make_query_record,
)
from avito_candidate_gen.ranking import DEFAULT_WEIGHTS, boost_score, rerank_top_k
from avito_candidate_gen.metrics import recall_at_50

TOP_K = 200        # глубина пула кандидатов
BATCH_SIZE = 2000  # запросов на один вызов retrieve_batch

PROCESSED_TRAIN = DATA_PROCESSED_DIR / "train_processed.parquet"
TITLE_CACHE = DATA_PROCESSED_DIR / "title_lemmas.parquet"
FEATURES_CACHE = DATA_PROCESSED_DIR / "val_features_top200.parquet"

### Данные

Читаем кэш предобработки **только с нужными колонками**

In [15]:
if not PROCESSED_TRAIN.exists():
    raise FileNotFoundError(
        "Кэш предобработки не найден — сначала выполните 02_baseline.ipynb"
    )

COLUMNS = [
    "search_query",
    "search_location_id",
    "search_infm_params_text",
    "item_id",
    "item_title_raw",
    "item_location_id",
    "item_infm_params_text",
    "item_rating",
    "item_rating_reviews_count",
    "item_text_processed",
]

df = pd.read_parquet(PROCESSED_TRAIN, columns=COLUMNS)
df_train, df_val = train_val_split(df)
print(f"train: {len(df_train):,} строк | val: {len(df_val):,} строк")

train: 403,091 строк | val: 94,582 строк


### Корпус и лемматизация заголовков

`item_title_lemma` нужна для признака `title_coverage` (и позже — для BM25-индекса по заголовкам). Заголовки дублируются на 39%, поэтому лемматизируем только уникальные и кэшируем результат на диск.

In [16]:
corpus = df.drop_duplicates(subset="item_id").reset_index(drop=True)
corpus = add_title_lemma_column(corpus, TITLE_CACHE)
print(f"Корпус: {len(corpus):,} уникальных объявлений")
corpus.head(3)

Читаю кэш заголовков: /Users/xxxogan/study/avito_test/avito-candidate-gen/data/processed/title_lemmas.parquet
Корпус: 344,825 уникальных объявлений


,search_query,search_location_id,search_infm_params_text,item_id,item_title_raw,item_location_id,item_infm_params_text,item_rating,item_rating_reviews_count,item_text_processed,item_title_lemma
0,скупка телевизоров,652430,,e8b685dffe1a408e,Скупка б/у техники,652430,Вид услуги Место оказания услуг Первомайский п...,4.989011,91.0,скупка б у техника скупка б у техника скупать ...,скупка б у техника
1,автоподбор,640860,Рейтинг пользователя 4 звезды и выше,92e1b0446f827b59,Автоподбор Разовый осмотр автомобиля,640860,Вид услуги Место оказания услуг Нижний Новгоро...,4.982684,462.0,автоподбор разовый осмотр автомобиль автоподбо...,автоподбор разовый осмотр автомобиль
2,баня на дровах,653240,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",624846856ce81d69,"Баня на дровах ""Прованс"" на Цветочной",653240,"Вид услуги Красота, здоровье Место оказания ус...",NaN,NaN,баня на дрова прованс на цветочный баня на дро...,баня на дрова прованс на цветочный


### Val-запросы, релевантные объявления, популярность

In [17]:
# Популярность = число кликов по объявлению
popularity = df_train.groupby("item_id").size()

# Признаки val-запросов
val_queries = df_val.groupby("search_query").first().reset_index()

val_queries["query_text"] = val_queries["search_query"].apply(build_query_text)
print(f"Лемматизировано {len(val_queries):,} val-запросов")

# Релевантные объявления на каждый val-запрос
relevant: dict[str, set[str]] = {
    str(q): set(items)
    for q, items in df_val.groupby("search_query")["item_id"].apply(set).items()
}

Лемматизировано 14,906 val-запросов


## Матрица признаков на BM25 top-200

Индекс по корпусу 345K + батчевый поиск по ~15K val-запросов

- `TOP_K = 200` — глубина пула;
- признаки собираются сразу в матрицу; строковые колонки (`query`, `item_id`) хранятся как `Categorical`;
- результат сохраняется в `val_features_top200.parquet`.

In [18]:
def build_features_dataset() -> pd.DataFrame:
    """
    BM25 top-200 на каждый val-запрос
    """
    retriever = BM25Retriever.from_dataframe(corpus)
    fb = FeatureBuilder(corpus, popularity=popularity)
    print(f"BM25-индекс и FeatureBuilder построены")

    nq = len(val_queries)
    n_rows = nq * TOP_K
    feat_all = np.empty((n_rows, len(FEATURE_COLUMNS)), dtype=np.float32)
    pos_all = np.empty(n_rows, dtype=np.int32)
    query_idx = np.empty(n_rows, dtype=np.int32)

    for start in range(0, nq, BATCH_SIZE):
        batch = val_queries.iloc[start : start + BATCH_SIZE]
        pos_batch, score_batch = retriever.retrieve_batch(
            batch["query_text"].tolist(), top_k=TOP_K
        )
        for j in range(len(batch)):
            assert len(pos_batch[j]) == TOP_K, "bm25s вернул меньше TOP_K кандидатов"
            row = batch.iloc[j]
            qrec = make_query_record(row, row["query_text"])
            gi = (start + j) * TOP_K
            feat_all[gi : gi + TOP_K] = fb.build_arrays(
                qrec, pos_batch[j], score_batch[j]
            )
            pos_all[gi : gi + TOP_K] = pos_batch[j]
            query_idx[gi : gi + TOP_K] = start + j
        done = min(start + BATCH_SIZE, nq)
        print(f"  {done:,}/{nq:,} запросов", flush=True)

    # Освобождение памяти перед сборкой таблицы
    del retriever
    gc.collect()

    features = pd.DataFrame(feat_all, columns=FEATURE_COLUMNS)
    features["query"] = pd.Categorical.from_codes(
        query_idx, categories=val_queries["search_query"].to_numpy()
    )
    features["item_id"] = pd.Categorical.from_codes(
        pos_all, categories=corpus["item_id"].to_numpy()
    )
    return features

In [19]:
if FEATURES_CACHE.exists():
    print(f"Читаю кэш признаков: {FEATURES_CACHE}")
    features = pd.read_parquet(FEATURES_CACHE)
else:
    features = build_features_dataset()
    features.to_parquet(FEATURES_CACHE)
    print(f"Сохранено в {FEATURES_CACHE}")

print(f"Матрица: {len(features):,} строк = {len(val_queries):,} запросов x {TOP_K} кандидатов")
features.head(3)

Читаю кэш признаков: /Users/xxxogan/study/avito_test/avito-candidate-gen/data/processed/val_features_top200.parquet
Матрица: 2,981,200 строк = 14,906 запросов x 200 кандидатов


,bm25_norm,loc_match,params_overlap,has_filter,title_coverage,rating_norm,rating_missing,reviews_norm,pop_norm,query,item_id
0,1.000000,0.0,0.0,0.0,0.666667,0.997315,0.0,0.540170,0.134506,"""промывка радиатора автомобиля без снятия любе...",4965e6b5c378ba7b
1,0.997172,0.0,0.0,0.0,0.666667,0.957447,0.0,0.417334,0.000000,"""промывка радиатора автомобиля без снятия любе...",97b19fef19b7f1cc
2,0.989213,0.0,0.0,0.0,0.333333,0.898361,0.0,0.444924,0.134506,"""промывка радиатора автомобиля без снятия любе...",aae3c568bf883ef9


## Реранк

Скор кандидата — взвешенная сумма признаков (`ranking.boost_score`).

`DEFAULT_WEIGHTS`: BM25 остаётся основным сигналом (вес 1.0), бусты заведомо меньше — они не ломают лексический порядок, но способны поднять кандидата со 100–300 места в топ-50. Сортировка устойчивая: при равенстве скора сохраняется BM25-порядок.

In [20]:
# Контроль: веса «только BM25» должны точно воспроизвести baseline 0.3534
bm25_only = dict.fromkeys(FEATURE_COLUMNS, 0.0) | {"bm25_norm": 1.0}
pred_bm25 = rerank_top_k(features, bm25_only, k=50)
score_bm25 = recall_at_50(pred_bm25, relevant)
print(f"Контроль (чистый BM25 top-50): {score_bm25:.4f}   [в baseline было 0.3534]")

Контроль (чистый BM25 top-50): 0.3534   [в baseline было 0.3534]


Контроль совпал с baseline до четвёртого знака: матрица признаков и реранк сами по себе ничего не ломают и полностью сохраняют BM25-порядок. Значит, приросты ниже — от бустов, а не от артефактов пайплайна.

In [21]:
pred_default = rerank_top_k(features, DEFAULT_WEIGHTS, k=50)
score_default = recall_at_50(pred_default, relevant)
print(f"DEFAULT_WEIGHTS: recall@50 = {score_default:.4f}")

DEFAULT_WEIGHTS: recall@50 = 0.4409


## Абляция весов

Каждый буст по отдельности поверх чистого BM25 — видно, какой сигнал реально работает, а какой шумит.

In [22]:
def evaluate(weights: dict) -> float:
    """recall@50 для данных весов (реранк всей закэшированной матрицы)"""
    return recall_at_50(rerank_top_k(features, weights, k=50), relevant)


base = dict.fromkeys(FEATURE_COLUMNS, 0.0) | {"bm25_norm": 1.0}

experiments = {
    "bm25 only (контроль)": base,
    "+ loc_match 0.15": base | {"loc_match": 0.15},
    "+ loc_match 0.50": base | {"loc_match": 0.50},
    "+ params_overlap 0.15": base | {"params_overlap": 0.15},
    "+ params_overlap 0.50": base | {"params_overlap": 0.50},
    "+ title_coverage 0.25": base | {"title_coverage": 0.25},
    "+ title_coverage 0.50": base | {"title_coverage": 0.50},
    "+ pop_norm 0.10": base | {"pop_norm": 0.10},
    "+ pop_norm 0.30": base | {"pop_norm": 0.30},
    "+ rating_norm 0.05": base | {"rating_norm": 0.05},
    "DEFAULT_WEIGHTS": DEFAULT_WEIGHTS,
}

results = {}
for name, w in experiments.items():
    results[name] = evaluate(w)
    print(f"{name:28s} recall@50 = {results[name]:.4f}", flush=True)

bm25 only (контроль)         recall@50 = 0.3534
+ loc_match 0.15             recall@50 = 0.4455
+ loc_match 0.50             recall@50 = 0.4546
+ params_overlap 0.15        recall@50 = 0.3593
+ params_overlap 0.50        recall@50 = 0.3642
+ title_coverage 0.25        recall@50 = 0.3713
+ title_coverage 0.50        recall@50 = 0.3683
+ pop_norm 0.10              recall@50 = 0.3436
+ pop_norm 0.30              recall@50 = 0.3240
+ rating_norm 0.05           recall@50 = 0.3523
DEFAULT_WEIGHTS              recall@50 = 0.4409


In [23]:
results_table = pd.Series(results, name="recall@50").sort_values(ascending=False)
results_table.to_frame()

,recall@50
+ loc_match 0.50,0.454559
+ loc_match 0.15,0.445451
DEFAULT_WEIGHTS,0.440900
+ title_coverage 0.25,0.371299
+ title_coverage 0.50,0.368300
+ params_overlap 0.50,0.364195
+ params_overlap 0.15,0.359279
bm25 only (контроль),0.353368
+ rating_norm 0.05,0.352282
+ pop_norm 0.10,0.343616


Выводы абляции:

- `loc_match` — главный сигнал: +0.10 к recall соло. Поиск сильно локализован (83% релевантных пар — та же локация).
- `params_overlap` и `title_coverage` — небольшой, но стабильный плюс; сильнее работают в комбинации, чем поодиночке.
- `pop_norm` вредит (−0.01…−0.03): буст популярности вытаскивает общие головные объявления и выталкивает точные совпадения, которые пользователь реально выбрал. `rating_norm` ≈ 0.

Популярность/рейтинг из линейного буста исключаем, но из матрицы признаков не убираем — обучаемый реранкер сможет найти им нелинейное применение.

### Лучшая комбинация

In [24]:
# Подбор весов сеткой по закэшированной матрице:
# sweep loc_match -> сетка title * params -> проверка мелких rating/pop сверху.
# - loc_match 0.5: при весе >= 0.75 буст превращается в почти жёсткий
#   фильтр по локации и начинает терять 17% релевантных объявлений из других локаций;
# - params_overlap сильнее (0.75), чем в одиночной абляции — раскрывается в комбинации;
# - популярность/рейтинг исключены: сверху ничего не добавляют (см. выводы абляции).
best_weights = dict.fromkeys(FEATURE_COLUMNS, 0.0) | {
    "bm25_norm": 1.0,
    "loc_match": 0.5,
    "title_coverage": 0.25,
    "params_overlap": 0.75,
}

best_score = evaluate(best_weights)
print(f"Best recall@50: {best_score:.4f}")

Best recall@50: 0.4658


In [25]:
experiment_row = pd.DataFrame(
    [
        {
            "approach": "boost_rerank_bm25_top300",
            "params": str(best_weights),
            "recall_at_50": best_score,
            "comment": "linear rerank of BM25 top-200: loc/params/title/rating/pop",
        }
    ]
)
exp_path = EXPERIMENTS_DIR / "experiments.csv"
experiment_row.to_csv(exp_path, mode="a", header=not exp_path.exists(), index=False)
print(f"Записано в {exp_path}")

Записано в /Users/xxxogan/study/avito_test/avito-candidate-gen/experiments/experiments.csv


## Ручная проверка

Запрос, на котором baseline промахнулся («промывка радиатора ... люберцы»): посмотрим, куда после бустов попало релевантное объявление и какие признаки его тянут.

In [26]:
# В тексте запроса буквально присутствуют кавычки — пользователь ввёл их в поисковую строку
q = '"промывка радиатора автомобиля без снятия люберцы"'

if q in relevant:
    sub = features[features["query"] == q].copy()
    sub["score"] = boost_score(sub, best_weights)
    rel_ids = relevant[q]

    top10 = sub.sort_values("score", ascending=False).head(10)
    top10["is_relevant"] = top10["item_id"].isin(rel_ids)
    cols = ["item_id", "score", "is_relevant", "bm25_norm", "loc_match", "title_coverage"]
    print("Топ-10 после реранка:")
    print(top10[cols].to_string(index=False))

    row = sub[sub["item_id"].isin(rel_ids)]
    if len(row):
        rank = int((sub["score"] > row["score"].iloc[0]).sum()) + 1
        print(f"\nПозиция релевантного объявления после реранка: {rank} / {TOP_K}")
    else:
        print("\nРелевантного объявления нет даже в топ-300 BM25 — пул его не покрыл")
else:
    print("Запроса нет в val-сплите")

Топ-10 после реранка:
         item_id    score  is_relevant  bm25_norm  loc_match  title_coverage
984907f2c811cc34 1.599236        False   0.974236        1.0        0.500000
98870c8ea8df9e23 1.519980        False   0.853314        1.0        0.666667
1b7199d1a15c5d12 1.497051        False   0.913718        1.0        0.333333
c5c45dc436e5705e 1.461819        False   0.878486        1.0        0.333333
6b57037005616e98 1.452271        False   0.910604        1.0        0.166667
44ab046a58706d19 1.444989        False   0.903322        1.0        0.166667
e5b9a64efa7b15e7 1.441104        False   0.816104        1.0        0.500000
3599a50852d53cdb 1.389063        False   0.805729        1.0        0.333333
8254bd2beb23ffa6 1.385098        False   0.801765        1.0        0.333333
cf50b80770d6537b 1.369423        False   0.827756        1.0        0.166667

Позиция релевантного объявления после реранка: 13 / 200


## Выводы

- Линейный реранк пула BM25 top-200 поднял recall@50 с **0.3534 до 0.4658** (+32% относительных) — без единой обученной модели, только на признаках, доступных в момент запроса.
- Главный сигнал — локация (+0.10 соло). Жёстким фильтром её делать нельзя: 17% релевантных объявлений находятся в других локациях, поэтому вес буста умеренный (0.5).
- Фильтры поиска и покрытие заголовка в комбинации дают ещё ~+0.011. Популярность и рейтинг в линейной формуле не работают (популярность вредит) — оставляем эти признаки для обучаемого реранкера.
- Взято ~85% потолка пула (recall@200 ≈ 0.548). Дальнейший рост — это расширение пула и новые источники retrieval (BM25 по заголовкам, char n-gram для опечаток), затем обучаемый реранкер.